# Chapter 2 (Extra Credit) — Train an adapter on top of MedImageInsight embeddings

Inputs: `embeddings.jsonl` + `embeddings.meta.json` from notebook 01.

We train two small classifier heads (SVM, MLP) on the frozen MI2 features, tune each
with a small Optuna search, and save them as `adapter_svm/weights.joblib` and `adapter_mlp/weights.joblib`.
Each pickle bundles:

- the fitted estimator,
- the class list (label order),
- the preprocessing recipe used to make the embeddings,
- per-class operating points (rule-in / rule-out thresholds).

The MCP `adapter_classify` tool is self-contained — it loads the pickle, embeds
the image with the matching recipe, and returns positive / possible / negative
calls. The downstream agent eval picks which adapter to use.

## 1. Setup

In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
EXTRA_CREDIT_DIR = NOTEBOOK_DIR.parent
CHAPTER_DIR = EXTRA_CREDIT_DIR.parent
CHAPTERS_DIR = CHAPTER_DIR.parent
WORKSHOP_ROOT = CHAPTERS_DIR.parent
EMB_DIR = EXTRA_CREDIT_DIR / "data/embeddings"
MODELS_DIR = EXTRA_CREDIT_DIR / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

EMB_PATH = EMB_DIR / "embeddings.jsonl"
META_PATH = EMB_DIR / "embeddings.meta.json"

print(f"embeddings: {EMB_PATH}")
print(f"meta:       {META_PATH}")
print(f"models:     {MODELS_DIR}")


## 2. Load embeddings + meta

In [2]:
with EMB_PATH.open() as f:
    rows = [json.loads(line) for line in f]
df = pd.DataFrame(rows)

X = np.asarray(df['embedding'].tolist(), dtype=np.float32)
y_str = df['label'].to_numpy()

meta = json.loads(META_PATH.read_text())
labels = meta['labels']
label_to_idx = {lbl: i for i, lbl in enumerate(labels)}
y = np.array([label_to_idx[lbl] for lbl in y_str])

print(f"X: {X.shape}, y: {y.shape}, classes: {len(labels)}")
print("class counts:")
print(pd.Series(y_str).value_counts().sort_index())


X: (1500, 1024), y: (1500,), classes: 15
class counts:
Atelectasis           100
Cardiomegaly          100
Consolidation         100
Edema                 100
Effusion              100
Emphysema             100
Fibrosis              100
Hernia                100
Infiltration          100
Mass                  100
No Finding            100
Nodule                100
Pleural_Thickening    100
Pneumonia             100
Pneumothorax          100
Name: count, dtype: int64


## 2a. Hold out the 210 eval set

The workshop's downstream agent eval runs over `internal-code/data/eval_set_210.jsonl`.
Those images are part of the 1500 we embedded — so to keep CV honest, pull them out
and reserve them as a **held-out test set**. CV / Optuna / final adapter fit only ever
see the trainval pool (~1290).

The adapter is then scored on the 210 once at the end. That number is comparable to the
agent's downstream eval and to chapter 1 / chapter 2 results.


In [3]:
EVAL_SET_PATH = WORKSHOP_ROOT / "internal-code/data/eval_set_210.jsonl"

with EVAL_SET_PATH.open() as f:
    eval_ids = {json.loads(line)["id"] for line in f}
print(f"eval set: {len(eval_ids)} ids")

# Each embedding row's image_path ends with the filename (e.g. .../Hernia/00012003_000.png).
# Match by basename so we don't depend on absolute paths.
basenames = df["image_path"].map(lambda p: Path(p).name)
holdout_mask = basenames.isin(eval_ids).to_numpy()

X_test = X[holdout_mask]
y_test = y[holdout_mask]
y_test_str = y_str[holdout_mask]

X_tv = X[~holdout_mask]
y_tv = y[~holdout_mask]
y_tv_str = y_str[~holdout_mask]

print(f"holdout (test): {X_test.shape[0]} rows  (expected {len(eval_ids)})")
print(f"trainval:       {X_tv.shape[0]} rows")

assert X_test.shape[0] == len(eval_ids), "some eval ids missing from embeddings"

# From here on, X / y refer to the trainval pool. Test set is X_test / y_test.
X, y, y_str = X_tv, y_tv, y_tv_str


eval set: 210 ids
holdout (test): 210 rows  (expected 210)
trainval:       1290 rows


## 3. Zero-shot baseline

Wrap MI2's CLIP-style zero-shot as a sklearn-compatible classifier so it slots into
`cross_val_score` alongside the trained heads. Text embeddings are cached to
`text_embeddings.jsonl` so re-runs are fully offline.


In [4]:
import sys, os
from sklearn.base import BaseEstimator, ClassifierMixin

MI2_SERVER_DIR = WORKSHOP_ROOT / "chapters/chapter2-foundation-models/answer-key/mcp/medimageinsight-server"
TEXT_EMB_PATH = EMB_DIR / "text_embeddings.jsonl"
os.environ.setdefault("MI2_ENV_FILE", str(MI2_SERVER_DIR / ".env"))
sys.path.insert(0, str(MI2_SERVER_DIR))
from workshop import mi2_client  # noqa: E402


def _load_text_emb_cache(path):
    """Return {prompt: (vec, scaling)} or {} if missing."""
    if not path.exists():
        return {}
    out = {}
    with path.open() as f:
        for line in f:
            r = json.loads(line)
            out[r["prompt"]] = (np.asarray(r["text_features"], dtype=np.float32), float(r["scaling_factor"]))
    return out


def _save_text_emb_cache(path, prompts, feats, scaling):
    with path.open("w") as f:
        for prompt, vec in zip(prompts, feats):
            print(json.dumps({"prompt": prompt, "scaling_factor": scaling, "text_features": vec.tolist()}), file=f)


class MI2ZeroShot(ClassifierMixin, BaseEstimator):
    """sklearn-compatible CLIP-style zero-shot over cached MI2 embeddings.
    Text embeddings are cached to TEXT_EMB_PATH (JSONL, one row per prompt) so
    re-runs are offline. fit() is a no-op so it slots into cross_val_score.
    """
    _estimator_type = "classifier"

    def __init__(self, labels, template="x-ray chest anteroposterior {}"):
        self.labels = labels
        self.template = template
        prompts = [template.format(lbl) for lbl in labels]

        cache = _load_text_emb_cache(TEXT_EMB_PATH)
        missing = [p for p in prompts if p not in cache]
        if missing:
            print(f"text-embed cache miss for {len(missing)} prompts; calling MI2 endpoint...")
            resp = mi2_client._score([[None, t] for t in missing])
            new_feats = [np.squeeze(np.array(r["text_features"])) for r in resp]
            new_scaling = float(np.exp(resp[0]["scaling_factor"]))
            for p, v in zip(missing, new_feats):
                cache[p] = (v, new_scaling)
            # rewrite cache with all prompts (canonical order = current label set)
            all_prompts = list(cache.keys())
            all_feats = np.stack([cache[p][0] for p in all_prompts])
            _save_text_emb_cache(TEXT_EMB_PATH, all_prompts, all_feats, new_scaling)
            print(f"saved text embeddings to {TEXT_EMB_PATH}")
        else:
            print(f"text embeddings loaded from cache ({TEXT_EMB_PATH.name}, {len(prompts)} prompts)")

        self._text_feats = np.stack([cache[p][0] for p in prompts])
        # all rows from one MI2 call share scaling_factor; use the first prompt's
        self._scaling = cache[prompts[0]][1]
        self.classes_ = np.arange(len(labels))

    def fit(self, X, y=None):
        self.classes_ = np.arange(len(self.labels))
        return self

    def predict_proba(self, X):
        logits = self._scaling * (X @ self._text_feats.T)
        logits = logits - logits.max(axis=1, keepdims=True)
        exp = np.exp(logits)
        return exp / exp.sum(axis=1, keepdims=True)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

    def __sklearn_tags__(self):
        tags = super().__sklearn_tags__()
        tags.estimator_type = "classifier"
        tags.classifier_tags.multi_class = True
        return tags

zs = MI2ZeroShot(labels)
print(f"text_feats: {zs._text_feats.shape}, scaling: {zs._scaling:.2f}")


text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
text_feats: (15, 1024), scaling: 91.23


## 4. Tune adapter heads

We tune two heads — RBF SVM and an MLP — with a small Optuna search over the same
5-fold CV / mAUC-OvR objective. Vanilla / un-tuned variants are skipped: tuning lifts
both by a few points and the goal is to put the adapters' best foot forward, not
litigate hyperparameter defaults.

Final candidates: `zero_shot` (baseline), `svm`, `mlp`.


In [5]:
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
baseline_candidates = {"zero_shot": zs}
trainable_candidates = {}  # filled by the tuning sweeps below


In [6]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

def _build_svm(trial):
    return make_pipeline(
        StandardScaler(),
        SVC(
            kernel="rbf",
            C=trial.suggest_float("C", 1e-2, 1e2, log=True),
            gamma=trial.suggest_categorical("gamma", ["scale", "auto"]),
            probability=True,
            random_state=42,
        ),
    )

def _build_mlp(trial):
    n_layers = trial.suggest_int("n_layers", 1, 3)
    hidden = tuple(
        trial.suggest_int(f"h{i}", 32, 512, log=True) for i in range(n_layers)
    )
    return make_pipeline(
        StandardScaler(),
        MLPClassifier(
            hidden_layer_sizes=hidden,
            alpha=trial.suggest_float("alpha", 1e-5, 1e-1, log=True),
            learning_rate_init=trial.suggest_float("lr", 1e-4, 1e-2, log=True),
            early_stopping=True,
            validation_fraction=0.15,
            n_iter_no_change=10,
            max_iter=300,
            random_state=42,
        ),
    )

BUILDERS = {"svm": _build_svm, "mlp": _build_mlp}
N_TRIALS = 50

studies = {}
for name, builder in BUILDERS.items():
    print(f"\n=== {name} ===")
    def objective(trial, _b=builder):
        return cross_val_score(_b(trial), X, y, cv=cv, scoring="roc_auc_ovr", n_jobs=-1).mean()
    study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)
    studies[name] = study
    print(f"  best mAUC: {study.best_value:.4f}")
    print(f"  best params: {study.best_params}")



=== svm ===


  0%|          | 0/50 [00:00<?, ?it/s]

  best mAUC: 0.8202
  best params: {'C': 0.576855557046952, 'gamma': 'scale'}

=== mlp ===


  0%|          | 0/50 [00:00<?, ?it/s]

  best mAUC: 0.8127
  best params: {'n_layers': 2, 'h0': 203, 'h1': 314, 'alpha': 0.00013616418646632394, 'lr': 0.00032941553241892796}


In [7]:
# Build the tuned candidates from each study's best_params, then CV-score the full lineup.
for name, study in studies.items():
    trainable_candidates[name] = BUILDERS[name](optuna.trial.FixedTrial(study.best_params))

candidates = {**baseline_candidates, **trainable_candidates}

scores = {}
for name, est in candidates.items():
    s = cross_val_score(est, X, y, cv=cv, scoring="roc_auc_ovr", n_jobs=-1)
    scores[name] = s
    print(f"{name:12s}  mAUC = {s.mean():.4f} \u00b1 {s.std():.4f}")


text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
text embeddings loaded from cache (text_embeddings.jsonl, 15 prompts)
zero_shot     mAUC = 0.8339 ± 0.0083
svm           mAUC = 0.8202 ± 0.0087
mlp           mAUC = 0.8127 ± 0.0054


## 5. Operating points + save adapters

For each trainable head, get out-of-fold `predict_proba` on trainval via
`cross_val_predict` and pick **two** per-class thresholds:

- `youden_j` — `argmax(sens + spec - 1)` on the one-vs-rest ROC. Rule-in cut.
- `sens90` — most permissive threshold whose sensitivity is ≥ 0.90. Rule-out cut.

The MCP tool reports `positive` if the score ≥ `youden_j`, `negative` if ≤ `sens90`,
and `possible` in between. Each head is then refit on the full trainval pool and
pickled to its own `adapter_<name>/weights.joblib`, with a sidecar
`adapter_<name>/thresholds.json` for human inspection.

In [8]:
import json
from datetime import datetime, timezone
import joblib
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import roc_curve


def operating_points(y_true, proba, labels, target_sens=0.90):
    """Per-class rule-in (Youden J) and rule-out (sens >= target_sens) thresholds."""
    out = {}
    for i, lbl in enumerate(labels):
        y_i = (y_true == i).astype(int)
        fpr, tpr, thr = roc_curve(y_i, proba[:, i])
        # roc_curve returns thresholds in descending order; tpr/fpr in ascending order.
        j_idx = int((tpr - fpr).argmax())
        # sens90: highest threshold whose sensitivity is >= target_sens.
        # tpr is ascending, so the first index where tpr >= target corresponds
        # to the highest threshold (thresholds sorted descending alongside tpr ascending).
        ok = (tpr >= target_sens).nonzero()[0]
        sens_thr = float(thr[ok[0]]) if len(ok) else 0.0
        out[lbl] = {
            "youden_j": float(thr[j_idx]),
            "sens90": sens_thr,
            "sensitivity": float(tpr[j_idx]),
            "specificity": float(1 - fpr[j_idx]),
            "prevalence": float(y_i.mean()),
        }
    return out


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, est in trainable_candidates.items():
    proba_oof = cross_val_predict(est, X, y, cv=cv, method="predict_proba")
    op = operating_points(y, proba_oof, labels, target_sens=0.90)

    est.fit(X, y)
    bundle = {
        "model": est,
        "classes": labels,
        "labels": labels,
        "preprocess": meta["preprocess"],
        "embedding_dim": meta["embedding_dim"],
        "operating_points": op,
        "trained_on": {
            "dataset": meta["dataset"],
            "n_rows": int(len(X)),
            "head": name,
            "cv_mauc_mean": float(scores[name].mean()),
            "cv_mauc_std": float(scores[name].std()),
        },
        "created_utc": datetime.now(timezone.utc).isoformat(),
    }
    out_dir = MODELS_DIR / f"adapter_{name}"
    out_dir.mkdir(parents=True, exist_ok=True)
    joblib.dump(bundle, out_dir / "weights.joblib")
    (out_dir / "thresholds.json").write_text(json.dumps(op, indent=2))
    size_kib = (out_dir / "weights.joblib").stat().st_size / 1024
    print(f"saved {out_dir.name}/  (mAUC {scores[name].mean():.4f}, {size_kib:.1f} KiB)")


saved adapter_svm/  (mAUC 0.8202, 10203.3 KiB)
saved adapter_mlp/  (mAUC 0.8127, 3279.1 KiB)


## 6. Held-out test (210)

CV gave us a model-selection number on trainval. Now run every candidate against
the 210 the adapter never saw — that's the apples-to-apples number against the
agent's downstream eval.


In [9]:
from sklearn.metrics import roc_auc_score

print(f"{'candidate':12s}  cv_mauc           test_mauc  test_top1")
print("-" * 58)
for name, est in candidates.items():
    est.fit(X, y)  # X/y are now trainval only
    proba = est.predict_proba(X_test)
    test_mauc = roc_auc_score(y_test, proba, multi_class="ovr")
    test_top1 = (proba.argmax(axis=1) == y_test).mean()
    cv_s = scores[name]
    print(f"{name:12s}  {cv_s.mean():.4f} \u00b1 {cv_s.std():.4f}   {test_mauc:.4f}     {test_top1:.4f}")


candidate     cv_mauc           test_mauc  test_top1
----------------------------------------------------------
zero_shot     0.8339 ± 0.0083   0.8186     0.3571
svm           0.8202 ± 0.0087   0.8099     0.3619
mlp           0.8127 ± 0.0054   0.8036     0.3619
